# Placement pair-weight controlled decomposition

이 Notebook은 occupied-cell grouping에서 관측된 작은 MRR 하락을 **pair 구성과 weight 변화**로 분해한다.
새 feature나 estimator를 제안하는 실험이 아니다.

위에서 아래로 실행하면 다음을 설명할 수 있어야 한다.

- candidate, physical placement class, pair의 차이
- zero-difference pair가 목적함수에 주는 영향
- decision-balanced weight가 selected alias와 generator multiplicity에 의해 어떻게 달라지는지
- raw train candidate에 fit한 shared scaler와 26-fold LOGO가 leakage를 막는 방식
- V0~V4 delta가 순서가 고정된 controlled decomposition이며 Shapley attribution이 아닌 이유

권장 읽기 순서: **worked example → V0~V4 생성식 → 26-fold 결과 → sanity gate와 결론**.

## 1. 실험 질문

**왜 실행하는가:** 이전 cell-grouped 실험은 zero pair 제거, weight 재정규화, class multiplicity 제거, row 압축을 동시에 바꿨다.

**입력:** 고정된 26게임 Human snapshot과 기존 실험 수치.

**계산:** 다섯 variant를 순서대로 비교한다.

**읽는 법:** 각 인접 delta가 어느 변환에서 metric이 움직였는지 보여준다.

**보류 조건:** V0 또는 V4 baseline이 재현되지 않거나 fold/scaler 계약이 깨지면 attribution을 중단한다.

In [ ]:
EXPERIMENT_ID = "placement-pair-weight-decomposition-v1"
PROGRAM_GATE = "continue_experimenting"
VARIANT_NAMES = [
    "raw_baseline",
    "drop_zero_keep_weight",
    "drop_zero_renormalized",
    "class_balanced_expanded",
    "cell_grouped",
]
EXPERIMENT_CARD = {
    "question": (
        "Cell grouping의 작은 MRR 하락은 zero-difference 제거, selected-alias dilution 해소, "
        "non-selected class multiplicity 제거, 실제 row compression 중 어디에서 발생했는가?"
    ),
    "changedVariable": "pair construction and pair weight only",
    "primaryDiagnostic": "physical-class held-out MRR",
    "secondaryDiagnostic": "raw-candidate held-out metrics",
    "fixedFold": "26-fold leave-one-game-out",
    "possibleStatuses": [
        "pair_weight_tradeoff_explained",
        "pair_weight_tradeoff_partially_explained",
        "pair_weight_tradeoff_not_explained",
        "blocked_by_baseline_mismatch",
    ],
    "attributionLimit": "ordered controlled decomposition; not order-independent Shapley attribution",
}
assert PROGRAM_GATE == "continue_experimenting"
assert len(VARIANT_NAMES) == 5 and len(set(VARIANT_NAMES)) == 5
EXPERIMENT_CARD

## 2. 고정 조건과 환경

**왜 실행하는가:** representation 효과에 feature, scaler, regularization, estimator 변화가 섞이지 않게 한다.

**입력:** Python 환경과 고정 model contract.

**계산:** 환경 버전과 정확한 설정을 기록한다.

**읽는 법:** 모든 variant가 동일한 설정을 공유해야 한다.

**보류 조건:** 예상 환경이 아니면 수치 비교를 시작하지 않는다.

In [ ]:
import hashlib
import io
import json
import os
import platform
import tempfile
import warnings
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import scipy
import sklearn
from IPython.display import display
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

EXPECTED_ENVIRONMENT = {
    "python": "3.12.13",
    "numpy": "2.5.1",
    "pandas": "3.0.3",
    "scipy": "1.18.0",
    "scikitLearn": "1.9.0",
}
ACTUAL_ENVIRONMENT = {
    "python": platform.python_version(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scipy": scipy.__version__,
    "scikitLearn": sklearn.__version__,
    "system": platform.system(),
    "machine": platform.machine(),
}
MODEL_CONFIG = {
    "model": "LogisticRegression",
    "solver": "lbfgs",
    "fitIntercept": False,
    "classWeight": None,
    "tolerance": 1e-12,
    "maxIterations": 10000,
    "l2": 0.01,
    "cFormula": "1/(fullTrainDecisionCount*l2)",
    "symmetricBinarySamples": True,
}
TOLERANCES = {
    "weight": 1e-12,
    "baselineDisplayedMetricDecimals": 6,
    "equivalentCoefficient": 1e-8,
    "equivalentScore": 1e-7,
    "metricEffect": 1e-12,
}
assert {key: ACTUAL_ENVIRONMENT[key] for key in EXPECTED_ENVIRONMENT} == EXPECTED_ENVIRONMENT
display(pd.DataFrame([ACTUAL_ENVIRONMENT]))
display(pd.DataFrame([MODEL_CONFIG]))

## 3. Source snapshot checksum 검증

**왜 실행하는가:** live log가 아닌 immutable snapshot만 읽었음을 증명한다.

**입력:** project-relative 26게임 snapshot 네 파일.

**계산:** 실행 전 SHA-256과 UTF-8/JSON 형식을 검증한다.

**읽는 법:** 네 digest가 고정 계약과 모두 같아야 한다.

**보류 조건:** 파일 부재나 한 byte라도 차이가 있으면 즉시 중단한다.

In [ ]:
def find_repository_root(start: Path) -> Path:
    """Find the project root without persisting an absolute path."""
    resolved = start.resolve()
    for candidate in [resolved, *resolved.parents]:
        if (candidate / "package.json").is_file() and (candidate / "AGENTS.md").is_file():
            return candidate
    raise AssertionError("REPOSITORY_ROOT_NOT_FOUND")


def sha256_file(path: Path) -> str:
    """Return a streaming SHA-256 digest."""
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


ROOT = find_repository_root(Path.cwd())
DATASET_RELATIVE_DIR = Path("data/datasets/human-behavior-v2-snapshot-26g-20260717")
DATASET_DIR = ROOT / DATASET_RELATIVE_DIR
SOURCE_FILES = {
    name: DATASET_DIR / name
    for name in ["manifest.json", "train.jsonl", "validation.jsonl", "rejected.jsonl"]
}
EXPECTED_SOURCE_SHA256 = {
    "manifest.json": "51027be0ebad578f2c38bb8715aa491daa8f13208e3da551a5e4a2dd08bef088",
    "train.jsonl": "85034a66373add54324a56673cb99ed5854810e4299abcaf9c195ae8cbca631e",
    "validation.jsonl": "23b832d95a2057b7b29e7062286c58c68524fa8db00f00870e6076a0a846f58a",
    "rejected.jsonl": "06ede14119014e2b040f8a4d62e00532ca6eecae9424b93e9cc882394db8a85f",
}
assert not DATASET_RELATIVE_DIR.is_absolute() and ".." not in DATASET_RELATIVE_DIR.parts
assert all(path.is_file() for path in SOURCE_FILES.values())
SOURCE_SHA256_BEFORE = {name: sha256_file(path) for name, path in SOURCE_FILES.items()}
assert SOURCE_SHA256_BEFORE == EXPECTED_SOURCE_SHA256, SOURCE_SHA256_BEFORE
display(pd.DataFrame([
    {"datasetPath": DATASET_RELATIVE_DIR.as_posix(), "file": name, "sha256": digest}
    for name, digest in SOURCE_SHA256_BEFORE.items()
]))

## 4. JSONL 로딩과 schema 검증

**왜 실행하는가:** label과 source 계약이 깨진 row를 정상 학습 데이터로 오인하지 않게 한다.

**입력:** manifest, train, validation, rejected JSONL.

**계산:** strict UTF-8/JSON, schema v2, Human source, boolean label, decision당 selected 1개를 확인한다.

**읽는 법:** 26 game, 1,158 decision, 39,375 candidate가 보여야 한다.

**보류 조건:** count, label, source, 기존 split leakage가 하나라도 다르면 중단한다.

In [ ]:
def read_json_object(path: Path) -> dict[str, Any]:
    value = json.loads(path.read_bytes().decode("utf-8", errors="strict"))
    assert isinstance(value, dict), f"JSON_OBJECT_REQUIRED:{path.name}"
    return value


def read_jsonl_frame(path: Path) -> pd.DataFrame:
    text = path.read_bytes().decode("utf-8", errors="strict")
    lines = text.splitlines()
    assert lines, f"EMPTY_JSONL:{path.name}"
    for line_number, line in enumerate(lines, start=1):
        assert line.strip(), f"EMPTY_JSONL_LINE:{path.name}:{line_number}"
        assert isinstance(json.loads(line), dict), f"JSON_OBJECT_LINE_REQUIRED:{path.name}:{line_number}"
    frame = pd.read_json(io.StringIO(text), lines=True, dtype=False)
    assert len(frame) == len(lines), f"ROW_COUNT_MISMATCH:{path.name}"
    return frame


manifest = read_json_object(SOURCE_FILES["manifest.json"])
train_source = read_jsonl_frame(SOURCE_FILES["train.jsonl"])
validation_source = read_jsonl_frame(SOURCE_FILES["validation.jsonl"])
rejected_source = read_jsonl_frame(SOURCE_FILES["rejected.jsonl"])
REQUIRED_COLUMNS = {
    "datasetSchemaVersion", "gameId", "playerId", "pieceIndex", "source",
    "aiDifficulty", "pieceType", "candidatePlacement", "candidateResult",
    "candidateMetricsAfter", "metricDelta", "selectedBySource", "metadata",
}
for split_name, frame in [("train", train_source), ("validation", validation_source)]:
    assert not REQUIRED_COLUMNS.difference(frame.columns), f"MISSING_COLUMNS:{split_name}"
    assert (frame["datasetSchemaVersion"] == 2).all(), f"SCHEMA_MISMATCH:{split_name}"
    assert (frame["source"] == "human").all(), f"NON_HUMAN_SOURCE:{split_name}"
    assert frame["selectedBySource"].map(
        lambda value: isinstance(value, (bool, np.bool_))
    ).all(), f"NON_BOOLEAN_SELECTED:{split_name}"

train_games = set(train_source["gameId"])
validation_games = set(validation_source["gameId"])
assert train_games.isdisjoint(validation_games), "ORIGINAL_SPLIT_GAME_LEAKAGE"
assert len(train_source) == manifest["trainSampleCount"]
assert len(validation_source) == manifest["validationSampleCount"]
assert len(rejected_source) == manifest["rejectedPlacementLogCount"]
train_source = train_source.copy(deep=True).assign(originalSplit="train")
validation_source = validation_source.copy(deep=True).assign(originalSplit="validation")
candidates = pd.concat([train_source, validation_source], ignore_index=True, copy=True)
assert len(candidates) == manifest["sampleCountBySource"]["human"] == 39_375
assert candidates["gameId"].nunique() == manifest["totalGameCount"] == 26

In [ ]:
def codepoint_key(value: Any) -> tuple[int, ...]:
    """Return an explicit Unicode code-point ordering key."""
    return tuple(ord(character) for character in str(value))


def create_decision_key(row: pd.Series) -> str:
    return f"{row['gameId']}:{row['playerId']}:{int(row['pieceIndex'])}"


def create_placement_key(row: pd.Series) -> str:
    placement = row["candidatePlacement"]
    return (
        f"{placement['pieceType']}:{int(placement['rotation'])}:"
        f"{int(placement['row'])}:{int(placement['col'])}"
    )


candidates["decisionKey"] = [create_decision_key(row) for _, row in candidates.iterrows()]
candidates["placementKey"] = [create_placement_key(row) for _, row in candidates.iterrows()]
decision_contract = candidates.groupby("decisionKey", sort=False).agg(
    candidateCount=("placementKey", "size"),
    placementCount=("placementKey", "nunique"),
    selectedCount=("selectedBySource", "sum"),
    gameCount=("gameId", "nunique"),
)
assert len(decision_contract) == manifest["acceptedPlacementLogCount"] == 1_158
assert (decision_contract["candidateCount"] >= 2).all()
assert (decision_contract["placementCount"] == decision_contract["candidateCount"]).all()
assert (decision_contract["selectedCount"] == 1).all()
assert (decision_contract["gameCount"] == 1).all()
display(pd.DataFrame([{
    "games": candidates["gameId"].nunique(),
    "decisions": len(decision_contract),
    "candidates": len(candidates),
    "selectedCandidates": int(candidates["selectedBySource"].sum()),
}]))

## 5. 기존 16-feature 추출

**왜 실행하는가:** 이번 실험이 feature 실험으로 변질되지 않았음을 고정한다.

**입력:** 각 candidate의 즉시 결과와 metric delta.

**계산:** 기존 순서 그대로 16차원 vector를 만든다.

**읽는 법:** selected label, 승패, heuristic score, 입력 sequence는 feature에 없다.

**보류 조건:** allowlist 순서·길이 또는 finite 값 계약이 깨지면 중단한다.

In [ ]:
FEATURE_SCHEMA_VERSION = 1
FEATURE_NAMES = [
    "candidateResult.linesCleared", "candidateResult.gameOver",
    "metricDelta.aggregateHeight", "metricDelta.maxHeight",
    "metricDelta.holes", "metricDelta.bumpiness",
    *[f"metricDelta.columnHeights[{index}]" for index in range(10)],
]
FORBIDDEN_FEATURES = {
    "selectedBySource", "metadata.isWinner", "winnerPlayerId",
    "heuristicBaseline.score", "metadata.inputSequence", "gameId", "playerId",
}


def extract_feature_vector(row: pd.Series) -> np.ndarray:
    result, delta = row["candidateResult"], row["metricDelta"]
    vector = np.asarray([
        float(result["linesCleared"]), float(result["gameOver"]),
        float(delta["aggregateHeight"]), float(delta["maxHeight"]),
        float(delta["holes"]), float(delta["bumpiness"]),
        *[float(value) for value in delta["columnHeights"]],
    ], dtype=np.float64)
    assert vector.shape == (16,) and np.isfinite(vector).all()
    return vector


assert len(FEATURE_NAMES) == 16 and set(FEATURE_NAMES).isdisjoint(FORBIDDEN_FEATURES)
candidates["featureVector"] = [extract_feature_vector(row) for _, row in candidates.iterrows()]
ordered_indices = sorted(
    candidates.index,
    key=lambda index: (
        codepoint_key(candidates.at[index, "gameId"]),
        codepoint_key(candidates.at[index, "decisionKey"]),
        codepoint_key(candidates.at[index, "placementKey"]),
    ),
)
candidates = candidates.loc[ordered_indices].reset_index(drop=True)
candidates["candidateIndex"] = np.arange(len(candidates), dtype=np.int64)
candidates["isWinner"] = candidates["metadata"].map(lambda value: bool(value["isWinner"]))
GAME_IDS = sorted(candidates["gameId"].unique(), key=codepoint_key)
assert len(GAME_IDS) == 26
display(pd.DataFrame({"index": range(16), "feature": FEATURE_NAMES}))

## 6. Occupied-cell equivalence key

**왜 실행하는가:** generator 표현이 달라도 같은 네 칸을 차지하면 같은 physical placement다.

**입력:** piece type과 absolute occupied cells.

**계산:** cell을 (row, col) 순으로 정렬해 label-free class key를 만든다.

**읽는 법:** rotation, anchor, placement ID는 key에 없고 차이 자체는 허용된다.

**보류 조건:** 한 class 안에서 feature나 즉시 결과가 다르거나 selected class가 하나가 아니면 학습하지 않는다.

In [ ]:
EQUIVALENCE_KEY_FIELDS = ["candidatePlacement.pieceType", "candidatePlacement.cells"]
EQUIVALENCE_KEY_FORBIDDEN_FIELDS = {
    "selectedBySource", "rotation", "row", "col", "placementKey",
    "metadata.isWinner", "winnerPlayerId", "heuristicBaseline.score",
    "metadata.inputSequence", "foldMetric",
}


def create_equivalence_key(row: pd.Series) -> str:
    placement = row["candidatePlacement"]
    cells = sorted((int(cell["row"]), int(cell["col"])) for cell in placement["cells"])
    assert len(cells) == 4 and len(set(cells)) == 4
    encoded = ";".join(f"{row_value},{col_value}" for row_value, col_value in cells)
    return f"{placement['pieceType']}|{encoded}"


assert set(EQUIVALENCE_KEY_FIELDS).isdisjoint(EQUIVALENCE_KEY_FORBIDDEN_FIELDS)
candidates["equivalenceKey"] = [
    create_equivalence_key(row) for _, row in candidates.iterrows()
]


def build_equivalence_classes(frame: pd.DataFrame) -> pd.DataFrame:
    """Build deterministic, label-free membership; label only marks the selected class later."""
    records = []
    for decision_key in sorted(frame["decisionKey"].unique(), key=codepoint_key):
        decision = frame.loc[frame["decisionKey"] == decision_key]
        for class_key in sorted(decision["equivalenceKey"].unique(), key=codepoint_key):
            members = decision.loc[decision["equivalenceKey"] == class_key]
            member_indices = sorted(
                members["candidateIndex"].tolist(),
                key=lambda index: codepoint_key(frame.at[index, "placementKey"]),
            )
            records.append({
                "gameId": members["gameId"].iloc[0],
                "decisionKey": decision_key,
                "pieceType": members["pieceType"].iloc[0],
                "aiDifficulty": members["aiDifficulty"].iloc[0],
                "isWinner": bool(members["isWinner"].iloc[0]),
                "equivalenceKey": class_key,
                "representativeIndex": int(member_indices[0]),
                "memberIndices": [int(value) for value in member_indices],
                "multiplicity": int(len(member_indices)),
                "selectedCandidateCount": int(members["selectedBySource"].sum()),
                "selectedClass": bool(members["selectedBySource"].any()),
            })
    return pd.DataFrame(records)


class_frame = build_equivalence_classes(candidates)
assert len(class_frame) == 26_403
assert (class_frame.groupby("decisionKey")["selectedClass"].sum() == 1).all()
assert (candidates.groupby("decisionKey")["selectedBySource"].sum() == 1).all()
for _, class_row in class_frame.iterrows():
    members = candidates.loc[class_row["memberIndices"]]
    features = np.vstack(members["featureVector"])
    assert np.array_equal(features, np.repeat(features[:1], len(features), axis=0))
    for field in ["candidateResult", "candidateMetricsAfter", "metricDelta"]:
        encoded = members[field].map(
            lambda value: json.dumps(value, sort_keys=True, separators=(",", ":"))
        )
        assert encoded.nunique() == 1, f"EQUIVALENCE_RESULT_MISMATCH:{field}"
display(pd.DataFrame([{
    "candidateCount": len(candidates),
    "physicalClassCount": len(class_frame),
    "removedAliasCount": len(candidates) - len(class_frame),
}]))

## 7. Worked example

**왜 실행하는가:** 같은 decision의 학습 영향력이 variant마다 어떻게 달라지는지 손으로 확인한다.

**입력:** selected class multiplicity가 2 이상이고 non-selected multiplicity가 여러 값인 decision 중 code-point 순 첫 항목.

**계산:** raw candidate/class 수, alias 수와 V0~V4 pair weight를 펼친다.

**읽는 법:** V0의 zero alias가 weight를 소비하고, V3는 class 총 weight를 같게 하며, V4는 같은 weight로 row만 압축한다.

**보류 조건:** 각 variant의 합이 정의와 다르거나 예시 선택이 비결정적이면 중단한다.

In [ ]:
selected_class_rows = class_frame.loc[class_frame["selectedClass"]]
worked_options = []
for decision_key in sorted(
    selected_class_rows.loc[selected_class_rows["multiplicity"] >= 2, "decisionKey"],
    key=codepoint_key,
):
    classes = class_frame.loc[class_frame["decisionKey"] == decision_key]
    nonselected = classes.loc[~classes["selectedClass"], "multiplicity"].tolist()
    if len(set(nonselected)) >= 2:
        worked_options.append(decision_key)
WORKED_DECISION = worked_options[0]
worked_candidates = candidates.loc[candidates["decisionKey"] == WORKED_DECISION].copy()
worked_classes = class_frame.loc[class_frame["decisionKey"] == WORKED_DECISION].copy()
worked_selected_class = worked_classes.loc[worked_classes["selectedClass"]].iloc[0]
worked_nonselected = worked_classes.loc[~worked_classes["selectedClass"]]
assert WORKED_DECISION == (
    "5FBD427F-1784135419893:"
    "bb3aa6ad-faa6-4b60-922c-dfe09399f9c3:37"
)
WORKED_SUMMARY = {
    "decisionKey": WORKED_DECISION,
    "rawCandidateCount": int(len(worked_candidates)),
    "physicalClassCount": int(len(worked_classes)),
    "selectedClassMultiplicity": int(worked_selected_class["multiplicity"]),
    "zeroDifferencePairCount": int(worked_selected_class["multiplicity"] - 1),
    "nonSelectedClassMultiplicities": sorted(
        [int(value) for value in worked_nonselected["multiplicity"]]
    ),
}
assert WORKED_SUMMARY["rawCandidateCount"] == 19
assert WORKED_SUMMARY["physicalClassCount"] == 10
assert WORKED_SUMMARY["selectedClassMultiplicity"] == 2
assert WORKED_SUMMARY["nonSelectedClassMultiplicities"] == [1, 2, 2, 2, 2, 2, 2, 2, 2]
display(pd.DataFrame([WORKED_SUMMARY]))

## 8. V0~V4 pair와 weight 생성

**왜 실행하는가:** 동시에 바뀌었던 네 요소를 한 번에 하나씩 바꾼다.

**입력:** 한 decision의 selected raw candidate, unselected candidates, equivalence classes.

**계산:**
- V0: 모든 raw pair, 각 weight = 1/raw alternatives
- V1: zero row 제거, V0 weight 유지
- V2: informative raw pair만 남기고 1/informative pairs
- V3: physical class 총 weight 1/K, alias 하나당 1/(K×M)
- V4: class 대표 row 하나, weight 1/K

**읽는 법:** V1 합은 1 미만일 수 있고 V2~V4의 represented decision 합은 1이다.

**보류 조건:** zero 판별, class별 합, pairless 기록, deterministic ordering 중 하나라도 깨지면 중단한다.

In [ ]:
VARIANT_DEFINITIONS = {
    "raw_baseline": "all raw alternatives; weight=1/rawAlternativeCount",
    "drop_zero_keep_weight": "drop zero differences; keep weight=1/rawAlternativeCount",
    "drop_zero_renormalized": "drop zero differences; weight=1/informativeRawPairCount",
    "class_balanced_expanded": "retain raw aliases; weight=1/(nonSelectedClassCount*classMultiplicity)",
    "cell_grouped": "one representative per non-selected class; weight=1/nonSelectedClassCount",
}


def build_variant_pairs(
    frame: pd.DataFrame,
    scaled_by_index: dict[int, np.ndarray],
    variant_name: str,
) -> tuple[pd.DataFrame, list[str]]:
    """Create one controlled pair representation in deterministic decision/key order."""
    assert variant_name in VARIANT_NAMES
    records, pairless = [], []
    for decision_key in sorted(frame["decisionKey"].unique(), key=codepoint_key):
        group = frame.loc[frame["decisionKey"] == decision_key]
        selected = group.loc[group["selectedBySource"]]
        alternatives = group.loc[~group["selectedBySource"]].copy()
        assert len(selected) == 1 and not alternatives.empty
        selected_row = selected.iloc[0]
        selected_index = int(selected_row["candidateIndex"])
        selected_class = selected_row["equivalenceKey"]
        raw_count = len(alternatives)
        alternatives = alternatives.sort_values(
            "placementKey", key=lambda column: column.map(codepoint_key)
        )
        alternatives["isZeroDifference"] = alternatives["featureVector"].map(
            lambda value: bool(np.array_equal(value, selected_row["featureVector"]))
        )
        informative = alternatives.loc[~alternatives["isZeroDifference"]].copy()
        physical = informative.loc[informative["equivalenceKey"] != selected_class].copy()
        class_keys = sorted(physical["equivalenceKey"].unique(), key=codepoint_key)
        selected_rows: list[tuple[pd.Series, float]] = []
        if variant_name == "raw_baseline":
            selected_rows = [(row, 1.0 / raw_count) for _, row in alternatives.iterrows()]
        elif variant_name == "drop_zero_keep_weight":
            selected_rows = [(row, 1.0 / raw_count) for _, row in informative.iterrows()]
        elif variant_name == "drop_zero_renormalized":
            selected_rows = [(row, 1.0 / len(informative)) for _, row in informative.iterrows()]
        elif variant_name == "class_balanced_expanded":
            for class_key in class_keys:
                members = physical.loc[physical["equivalenceKey"] == class_key]
                weight = 1.0 / (len(class_keys) * len(members))
                selected_rows.extend((row, weight) for _, row in members.iterrows())
        else:
            for class_key in class_keys:
                members = physical.loc[physical["equivalenceKey"] == class_key]
                selected_rows.append((members.iloc[0], 1.0 / len(class_keys)))
        if not selected_rows:
            pairless.append(decision_key)
            continue
        for alternative_row, pair_weight in selected_rows:
            alternative_index = int(alternative_row["candidateIndex"])
            records.append({
                "decisionKey": decision_key,
                "selectedIndex": selected_index,
                "alternativeIndex": alternative_index,
                "alternativeClassKey": alternative_row["equivalenceKey"],
                "isZeroDifference": bool(alternative_row["isZeroDifference"]),
                "pairWeight": float(pair_weight),
                "difference": (
                    scaled_by_index[selected_index] - scaled_by_index[alternative_index]
                ),
            })
    return pd.DataFrame(records), pairless


def raw_index_map(frame: pd.DataFrame) -> dict[int, np.ndarray]:
    return {
        int(row["candidateIndex"]): row["featureVector"]
        for _, row in frame.iterrows()
    }

In [ ]:
worked_map = raw_index_map(worked_candidates)
worked_variant_rows = []
for variant_name in VARIANT_NAMES:
    pairs, pairless = build_variant_pairs(worked_candidates, worked_map, variant_name)
    assert not pairless
    for _, pair in pairs.iterrows():
        worked_variant_rows.append({
            "variant": variant_name,
            "alternativePlacement": candidates.at[pair["alternativeIndex"], "placementKey"],
            "alternativeClass": pair["alternativeClassKey"],
            "zeroDifference": pair["isZeroDifference"],
            "pairWeight": pair["pairWeight"],
        })
worked_variant_table = pd.DataFrame(worked_variant_rows)
worked_weight_summary = worked_variant_table.groupby("variant", sort=False).agg(
    pairCount=("pairWeight", "size"),
    decisionWeightSum=("pairWeight", "sum"),
    zeroPairCount=("zeroDifference", "sum"),
).reindex(VARIANT_NAMES).reset_index()
expected_worked = {
    "raw_baseline": (18, 1.0, 1),
    "drop_zero_keep_weight": (17, 17.0 / 18.0, 0),
    "drop_zero_renormalized": (17, 1.0, 0),
    "class_balanced_expanded": (17, 1.0, 0),
    "cell_grouped": (9, 1.0, 0),
}
for row in worked_weight_summary.to_dict(orient="records"):
    pair_count, total, zero_count = expected_worked[row["variant"]]
    assert row["pairCount"] == pair_count and row["zeroPairCount"] == zero_count
    assert abs(row["decisionWeightSum"] - total) <= TOLERANCES["weight"]
display(worked_weight_summary)
display(worked_variant_table)

## 9. 전체 variant pair와 weight 진단

**왜 실행하는가:** 모델 학습 전에 representation 자체의 count와 실제 weight 합을 확정한다.

**입력:** 전체 1,158 decision의 raw feature vector와 class membership.

**계산:** pair 수, zero 수, pairless decision, effective decision, symmetric sample, weight 합/min/max를 집계한다.

**읽는 법:** V0 38,217 pair, V4 25,245 pair여야 하고 V1만 decision weight가 1 미만일 수 있다.

**보류 조건:** 알려진 snapshot count 또는 weight 공식과 다르면 LOGO를 실행하지 않는다.

In [ ]:
def summarize_pair_frame(
    variant_name: str,
    pairs: pd.DataFrame,
    pairless: list[str],
    full_decision_count: int,
) -> dict[str, Any]:
    sums = pairs.groupby("decisionKey", sort=False)["pairWeight"].sum()
    return {
        "variant": variant_name,
        "rawPairCount": 38_217,
        "retainedPairCount": int(len(pairs)),
        "availableZeroDifferencePairCount": 1_010,
        "zeroDifferencePairCount": int(pairs["isZeroDifference"].sum()),
        "physicalClassPairCount": 25_245,
        "pairlessDecisionCount": int(len(pairless)),
        "trainDecisionCount": int(full_decision_count),
        "effectiveDecisionCount": int(len(sums)),
        "symmetricSampleCount": int(len(pairs) * 2),
        "sampleWeightTotal": float(sums.sum()),
        "minimumDecisionWeight": float(sums.min()),
        "maximumDecisionWeight": float(sums.max()),
        "unitWeightDecisionCount": int(np.isclose(
            sums.to_numpy(), 1.0, rtol=0.0, atol=TOLERANCES["weight"]
        ).sum()),
    }


FULL_VARIANT_PAIRS = {}
FULL_PAIRLESS = {}
PAIR_DIAGNOSTICS = {}
full_raw_map = raw_index_map(candidates)
for variant_name in VARIANT_NAMES:
    pairs, pairless = build_variant_pairs(candidates, full_raw_map, variant_name)
    FULL_VARIANT_PAIRS[variant_name] = pairs
    FULL_PAIRLESS[variant_name] = pairless
    PAIR_DIAGNOSTICS[variant_name] = summarize_pair_frame(
        variant_name, pairs, pairless, len(decision_contract)
    )

assert PAIR_DIAGNOSTICS["raw_baseline"]["retainedPairCount"] == 38_217
assert PAIR_DIAGNOSTICS["raw_baseline"]["zeroDifferencePairCount"] == 1_010
assert PAIR_DIAGNOSTICS["cell_grouped"]["retainedPairCount"] == 25_245
assert PAIR_DIAGNOSTICS["drop_zero_keep_weight"]["pairlessDecisionCount"] == 2
assert PAIR_DIAGNOSTICS["drop_zero_renormalized"]["pairlessDecisionCount"] == 2
assert PAIR_DIAGNOSTICS["class_balanced_expanded"]["pairlessDecisionCount"] == 2
assert PAIR_DIAGNOSTICS["cell_grouped"]["pairlessDecisionCount"] == 2

selected_multiplicity_distribution = (
    class_frame.loc[class_frame["selectedClass"], "multiplicity"]
    .value_counts().sort_index().rename_axis("multiplicity").reset_index(name="classCount")
)
nonselected_multiplicity_distribution = (
    class_frame.loc[~class_frame["selectedClass"], "multiplicity"]
    .value_counts().sort_index().rename_axis("multiplicity").reset_index(name="classCount")
)
display(pd.DataFrame(PAIR_DIAGNOSTICS.values()))
display(selected_multiplicity_distribution)
display(nonselected_multiplicity_distribution)

## 10. Shared scaler, symmetric sample, fixed estimator

**왜 실행하는가:** pair weight 외의 변수를 fold 안에서 완전히 공유한다.

**입력:** validation game을 제외한 raw train candidates.

**계산:** StandardScaler를 raw train candidate에 한 번 fit하고 모든 variant가 같은 mean/scale을 사용한다. 각 pair는 ±difference 두 sample로 만들며 pair weight를 절반씩 나눈다.

**읽는 법:** C의 분모는 pairless 여부와 무관하게 원래 fold의 전체 train decision 수다.

**보류 조건:** validation이 scaler/pair/weight에 들어가거나 실제 sklearn sample weight 합이 pair 정의와 다르면 중단한다.

In [ ]:
def fit_shared_scaler(
    train_frame: pd.DataFrame,
    validation_frame: pd.DataFrame,
) -> tuple[StandardScaler, dict[int, np.ndarray], dict[int, np.ndarray]]:
    train_values = np.vstack(train_frame["featureVector"])
    validation_values = np.vstack(validation_frame["featureVector"])
    scaler = StandardScaler(with_mean=True, with_std=True)
    train_scaled = scaler.fit_transform(train_values)
    validation_scaled = scaler.transform(validation_values)
    assert int(scaler.n_samples_seen_) == len(train_frame)
    train_map = {
        int(index): train_scaled[position]
        for position, index in enumerate(train_frame["candidateIndex"].tolist())
    }
    validation_map = {
        int(index): validation_scaled[position]
        for position, index in enumerate(validation_frame["candidateIndex"].tolist())
    }
    return scaler, train_map, validation_map


def create_symmetric_samples(
    pairs: pd.DataFrame,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, dict[str, Any]]:
    differences = np.vstack(pairs["difference"])
    pair_weights = pairs["pairWeight"].to_numpy(dtype=np.float64)
    count, feature_count = differences.shape
    x_values = np.empty((count * 2, feature_count), dtype=np.float64)
    labels = np.empty(count * 2, dtype=np.int64)
    sample_weights = np.empty(count * 2, dtype=np.float64)
    x_values[0::2], x_values[1::2] = differences, -differences
    labels[0::2], labels[1::2] = 1, 0
    sample_weights[0::2] = pair_weights / 2.0
    sample_weights[1::2] = pair_weights / 2.0
    decision_keys = np.repeat(pairs["decisionKey"].to_numpy(), 2)
    actual = pd.DataFrame({
        "decisionKey": decision_keys, "weight": sample_weights
    }).groupby("decisionKey", sort=False)["weight"].sum()
    expected = pairs.groupby("decisionKey", sort=False)["pairWeight"].sum()
    assert np.allclose(actual.loc[expected.index], expected, rtol=0.0, atol=TOLERANCES["weight"])
    assert np.array_equal(sample_weights[0::2], sample_weights[1::2])
    return x_values, labels, sample_weights, {
        "representedDecisionCount": int(len(actual)),
        "totalWeight": float(sample_weights.sum()),
        "minimumDecisionWeight": float(actual.min()),
        "maximumDecisionWeight": float(actual.max()),
    }


def fit_pairwise_model(
    pairs: pd.DataFrame,
    full_train_decision_count: int,
) -> tuple[LogisticRegression, dict[str, Any]]:
    x_values, labels, sample_weights, weight_check = create_symmetric_samples(pairs)
    c_value = 1.0 / (full_train_decision_count * MODEL_CONFIG["l2"])
    model = LogisticRegression(
        C=c_value,
        solver=MODEL_CONFIG["solver"],
        fit_intercept=MODEL_CONFIG["fitIntercept"],
        class_weight=MODEL_CONFIG["classWeight"],
        tol=MODEL_CONFIG["tolerance"],
        max_iter=MODEL_CONFIG["maxIterations"],
    )
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        model.fit(x_values, labels, sample_weight=sample_weights)
    convergence = [
        str(item.message) for item in caught
        if issubclass(item.category, ConvergenceWarning)
    ]
    unexpected = [
        f"{item.category.__name__}:{item.message}" for item in caught
        if not issubclass(item.category, ConvergenceWarning)
    ]
    assert not convergence, convergence
    assert int(model.n_iter_[0]) < MODEL_CONFIG["maxIterations"]
    return model, {
        "c": float(c_value),
        "iterations": int(model.n_iter_[0]),
        "convergenceWarnings": convergence,
        "unexpectedWarnings": unexpected,
        "sampleWeight": weight_check,
    }

## 11. Held-out raw/class metric

**왜 실행하는가:** 학습 representation과 평가 단위를 구분한다.

**입력:** held-out candidate의 shared-scaled feature와 fold coefficient.

**계산:** raw candidate와 physical class를 각각 canonical key로 tie-break해 rank를 구한다.

**읽는 법:** physical-class MRR가 primary diagnostic이고 raw metric은 과거 연속성용 secondary다.

**보류 조건:** scorer가 validation 외 row를 보거나 selected rank가 하나로 정해지지 않으면 중단한다.

In [ ]:
def summarize_predictions(predictions: list[dict[str, Any]]) -> dict[str, Any]:
    frame = pd.DataFrame(predictions)
    ranks = frame["selectedRank"].to_numpy(dtype=np.int64)
    return {
        "decisionCount": int(len(frame)),
        "top1Count": int((ranks == 1).sum()),
        "top3Count": int((ranks <= 3).sum()),
        "top1Accuracy": float(np.mean(ranks == 1)),
        "top3Accuracy": float(np.mean(ranks <= 3)),
        "mrr": float(np.mean(1.0 / ranks)),
        "meanSelectedRank": float(np.mean(ranks)),
        "tieAwareTop1Count": int(frame["selectedAtTopScore"].sum()),
        "topScoreTieDecisionCount": int((frame["topScoreTieCount"] > 1).sum()),
    }


def score_raw_candidates(
    validation_frame: pd.DataFrame,
    scaled_by_index: dict[int, np.ndarray],
    coefficients: np.ndarray,
) -> list[dict[str, Any]]:
    predictions = []
    for decision_key in sorted(validation_frame["decisionKey"].unique(), key=codepoint_key):
        group = validation_frame.loc[validation_frame["decisionKey"] == decision_key]
        rows = [{
            "key": row["placementKey"],
            "selected": bool(row["selectedBySource"]),
            "score": float(scaled_by_index[int(row["candidateIndex"])] @ coefficients),
        } for _, row in group.iterrows()]
        ranked = sorted(rows, key=lambda row: (-row["score"], codepoint_key(row["key"])))
        selected = next(row for row in rows if row["selected"])
        top_score = ranked[0]["score"]
        rank = next(position for position, row in enumerate(ranked, 1) if row["key"] == selected["key"])
        predictions.append({
            "gameId": group["gameId"].iloc[0],
            "decisionKey": decision_key,
            "selectedRank": rank,
            "selectedAtTopScore": selected["score"] == top_score,
            "topScoreTieCount": sum(row["score"] == top_score for row in rows),
            "selectedScore": selected["score"],
        })
    return predictions


def score_physical_classes(
    validation_frame: pd.DataFrame,
    scaled_by_index: dict[int, np.ndarray],
    coefficients: np.ndarray,
) -> list[dict[str, Any]]:
    predictions = []
    validation_decisions = set(validation_frame["decisionKey"])
    subset = class_frame.loc[class_frame["decisionKey"].isin(validation_decisions)]
    for decision_key in sorted(validation_decisions, key=codepoint_key):
        group = subset.loc[subset["decisionKey"] == decision_key]
        rows = [{
            "key": row["equivalenceKey"],
            "selected": bool(row["selectedClass"]),
            "score": float(scaled_by_index[int(row["representativeIndex"])] @ coefficients),
        } for _, row in group.iterrows()]
        ranked = sorted(rows, key=lambda row: (-row["score"], codepoint_key(row["key"])))
        selected = next(row for row in rows if row["selected"])
        top_score = ranked[0]["score"]
        rank = next(position for position, row in enumerate(ranked, 1) if row["key"] == selected["key"])
        predictions.append({
            "gameId": group["gameId"].iloc[0],
            "decisionKey": decision_key,
            "selectedRank": rank,
            "selectedAtTopScore": selected["score"] == top_score,
            "topScoreTieCount": sum(row["score"] == top_score for row in rows),
            "selectedScore": selected["score"],
        })
    return predictions

## 12. 26-fold LOGO

**왜 실행하는가:** 각 game을 한 번씩 완전히 held-out해 in-sample aggregate를 막는다.

**입력:** code-point 순 26 game ID.

**계산:** fold당 scaler 한 개, V0~V4 model 다섯 개, raw/class held-out prediction을 만든다.

**읽는 법:** 26/26 convergence, leakage 0, scaler fit count=raw train candidates가 필수다.

**보류 조건:** fold, scaler mean/scale, C가 variant 사이에서 다르면 중단한다.

In [ ]:
FOLD_RESULTS = []
HELD_OUT = {
    variant: {"rawCandidate": [], "physicalClass": []}
    for variant in VARIANT_NAMES
}
for fold_number, validation_game_id in enumerate(GAME_IDS, start=1):
    validation_mask = candidates["gameId"] == validation_game_id
    train_fold = candidates.loc[~validation_mask].copy()
    validation_fold = candidates.loc[validation_mask].copy()
    assert set(train_fold["gameId"]).isdisjoint(set(validation_fold["gameId"]))
    scaler, train_map, validation_map = fit_shared_scaler(train_fold, validation_fold)
    full_train_decision_count = int(train_fold["decisionKey"].nunique())
    fold_record = {
        "foldNumber": fold_number,
        "validationGameId": validation_game_id,
        "trainGameCount": int(train_fold["gameId"].nunique()),
        "trainDecisionCount": full_train_decision_count,
        "validationDecisionCount": int(validation_fold["decisionKey"].nunique()),
        "rawTrainCandidateCount": int(len(train_fold)),
        "scalerFitCandidateCount": int(scaler.n_samples_seen_),
        "scalerFitScope": "raw_train_candidates_only",
        "gameIdLeakageCount": 0,
        "sharedScalerAcrossVariants": True,
        "variants": {},
    }
    c_values = []
    for variant_name in VARIANT_NAMES:
        pairs, pairless = build_variant_pairs(train_fold, train_map, variant_name)
        model, fit = fit_pairwise_model(pairs, full_train_decision_count)
        c_values.append(fit["c"])
        raw_predictions = score_raw_candidates(validation_fold, validation_map, model.coef_[0])
        class_predictions = score_physical_classes(validation_fold, validation_map, model.coef_[0])
        HELD_OUT[variant_name]["rawCandidate"].extend(raw_predictions)
        HELD_OUT[variant_name]["physicalClass"].extend(class_predictions)
        fold_record["variants"][variant_name] = {
            "pairCount": int(len(pairs)),
            "pairlessDecisionCount": int(len(pairless)),
            "fit": fit,
            "coefficient": model.coef_[0].tolist(),
            "coefficientL2Norm": float(np.linalg.norm(model.coef_[0])),
            "rawCandidateMetrics": summarize_predictions(raw_predictions),
            "physicalClassMetrics": summarize_predictions(class_predictions),
            "rawPredictions": raw_predictions,
            "physicalClassPredictions": class_predictions,
        }
    assert len(set(c_values)) == 1
    assert fold_record["scalerFitCandidateCount"] == fold_record["rawTrainCandidateCount"]
    validation_matrix = np.vstack([
        validation_map[int(index)]
        for index in validation_fold["candidateIndex"].tolist()
    ])
    comparison_pairs = [
        *list(zip(VARIANT_NAMES, VARIANT_NAMES[1:])),
        ("raw_baseline", "cell_grouped"),
    ]
    fold_record["validationScoreMaximumAbsoluteDifferences"] = {}
    for left, right in comparison_pairs:
        left_coefficient = np.asarray(fold_record["variants"][left]["coefficient"])
        right_coefficient = np.asarray(fold_record["variants"][right]["coefficient"])
        left_scores = validation_matrix @ left_coefficient
        right_scores = validation_matrix @ right_coefficient
        fold_record["validationScoreMaximumAbsoluteDifferences"][
            f"{right}-minus-{left}"
        ] = float(np.max(np.abs(left_scores - right_scores)))
    FOLD_RESULTS.append(fold_record)

assert len(FOLD_RESULTS) == 26
assert sum(fold["gameIdLeakageCount"] for fold in FOLD_RESULTS) == 0
assert all(
    not fold["variants"][variant]["fit"]["convergenceWarnings"]
    for fold in FOLD_RESULTS for variant in VARIANT_NAMES
)

## 13. Raw-candidate와 physical-class 결과

**왜 실행하는가:** 평가 단위가 다른 metric을 섞어 해석하지 않게 한다.

**입력:** 각 fold의 held-out prediction만 사용한다.

**계산:** micro, fold macro/median/min/max와 MRR 최저 fold를 집계한다.

**읽는 법:** class 결과가 primary이며 raw 결과는 secondary다.

**보류 조건:** 어느 variant든 held-out decision 수가 1,158이 아니면 aggregate를 사용하지 않는다.

In [ ]:
METRIC_NAMES = ["top1Accuracy", "top3Accuracy", "mrr", "meanSelectedRank"]


def aggregate_variant_evaluation(
    variant_name: str,
    evaluation_name: str,
) -> dict[str, Any]:
    metric_field = (
        "rawCandidateMetrics" if evaluation_name == "rawCandidate"
        else "physicalClassMetrics"
    )
    fold_rows = []
    for fold in FOLD_RESULTS:
        metrics = fold["variants"][variant_name][metric_field]
        fold_rows.append({"validationGameId": fold["validationGameId"], **metrics})
    frame = pd.DataFrame(fold_rows)
    worst = min(
        fold_rows,
        key=lambda row: (row["mrr"], codepoint_key(row["validationGameId"])),
    )
    return {
        "micro": summarize_predictions(HELD_OUT[variant_name][evaluation_name]),
        "macro": {name: float(frame[name].mean()) for name in METRIC_NAMES},
        "median": {name: float(frame[name].median()) for name in METRIC_NAMES},
        "minimum": {name: float(frame[name].min()) for name in METRIC_NAMES},
        "maximum": {name: float(frame[name].max()) for name in METRIC_NAMES},
        "worstFold": {
            "validationGameId": worst["validationGameId"],
            "mrr": float(worst["mrr"]),
            "decisionCount": int(worst["decisionCount"]),
        },
    }


EVALUATION_RESULTS = {
    variant: {
        evaluation: aggregate_variant_evaluation(variant, evaluation)
        for evaluation in ["rawCandidate", "physicalClass"]
    }
    for variant in VARIANT_NAMES
}
for variant in VARIANT_NAMES:
    assert EVALUATION_RESULTS[variant]["rawCandidate"]["micro"]["decisionCount"] == 1_158
    assert EVALUATION_RESULTS[variant]["physicalClass"]["micro"]["decisionCount"] == 1_158

raw_table = pd.DataFrame([{
    "variant": variant,
    **EVALUATION_RESULTS[variant]["rawCandidate"]["micro"],
    "macroMrr": EVALUATION_RESULTS[variant]["rawCandidate"]["macro"]["mrr"],
} for variant in VARIANT_NAMES])
class_table = pd.DataFrame([{
    "variant": variant,
    **EVALUATION_RESULTS[variant]["physicalClass"]["micro"],
    "macroMrr": EVALUATION_RESULTS[variant]["physicalClass"]["macro"]["mrr"],
} for variant in VARIANT_NAMES])
display(raw_table)
display(class_table)

## 14. Coefficient, score, rank 비교

**왜 실행하는가:** metric count만 같아도 coefficient가 달라질 수 있으므로 representation equivalence를 직접 확인한다.

**입력:** 같은 fold의 인접 variant coefficient와 held-out selected score/rank.

**계산:** L2 norm, cosine, max coefficient/score 차이, raw/class rank 변경 수를 집계한다.

**읽는 법:** V0↔V1과 V3↔V4는 수치 tolerance 안에서 equivalent해야 한다.

**보류 조건:** V3↔V4가 다르면 optimizer 탓으로 추측하지 않고 pair identity와 weight를 먼저 조사한다.

In [ ]:
def compare_variants(left: str, right: str) -> dict[str, Any]:
    cosine_values, coefficient_differences, score_differences = [], [], []
    raw_rank_changes = class_rank_changes = 0
    left_l2, right_l2 = [], []
    for fold in FOLD_RESULTS:
        left_record, right_record = fold["variants"][left], fold["variants"][right]
        score_differences.append(
            fold["validationScoreMaximumAbsoluteDifferences"][f"{right}-minus-{left}"]
        )
        left_coef = np.asarray(left_record["coefficient"])
        right_coef = np.asarray(right_record["coefficient"])
        denominator = np.linalg.norm(left_coef) * np.linalg.norm(right_coef)
        cosine_values.append(float(left_coef @ right_coef / denominator))
        coefficient_differences.append(float(np.max(np.abs(left_coef - right_coef))))
        left_l2.append(float(np.linalg.norm(left_coef)))
        right_l2.append(float(np.linalg.norm(right_coef)))
        for evaluation_name, prediction_field in [
            ("rawCandidate", "rawPredictions"),
            ("physicalClass", "physicalClassPredictions"),
        ]:
            left_rows = left_record[prediction_field]
            right_rows = right_record[prediction_field]
            assert [row["decisionKey"] for row in left_rows] == [
                row["decisionKey"] for row in right_rows
            ]
            changes = sum(
                left_row["selectedRank"] != right_row["selectedRank"]
                for left_row, right_row in zip(left_rows, right_rows)
            )
            if evaluation_name == "rawCandidate":
                raw_rank_changes += changes
            else:
                class_rank_changes += changes
    return {
        "left": left,
        "right": right,
        "leftCoefficientL2Minimum": float(min(left_l2)),
        "leftCoefficientL2Maximum": float(max(left_l2)),
        "rightCoefficientL2Minimum": float(min(right_l2)),
        "rightCoefficientL2Maximum": float(max(right_l2)),
        "minimumCosineSimilarity": float(min(cosine_values)),
        "meanCosineSimilarity": float(np.mean(cosine_values)),
        "maximumAbsoluteCoefficientDifference": float(max(coefficient_differences)),
        "maximumAbsoluteValidationCandidateScoreDifference": float(max(score_differences)),
        "rawCandidateRankChangeCount": int(raw_rank_changes),
        "physicalClassRankChangeCount": int(class_rank_changes),
    }


COMPARISONS = {}
for left, right in zip(VARIANT_NAMES, VARIANT_NAMES[1:]):
    COMPARISONS[f"{right}-minus-{left}"] = compare_variants(left, right)
COMPARISONS["cell_grouped-minus-raw_baseline"] = compare_variants(
    "raw_baseline", "cell_grouped"
)
display(pd.DataFrame(COMPARISONS.values()))

## 15. Fold delta, multiplicity segmentation, pairless 진단

**왜 실행하는가:** aggregate의 작은 차이가 특정 fold나 selected multiplicity에만 의존하는지 확인한다.

**입력:** 인접 variant held-out physical-class rank와 game별 MRR.

**계산:** ordered effect, 개선/동일/하락 fold, worst fold, selected multiplicity segment를 만든다.

**읽는 법:** 각 effect delta의 합은 V4−V0이고, 이는 정해진 순서에만 유효하다.

**보류 조건:** 합계가 total delta와 맞지 않거나 pairless decision이 뜻하지 않게 빠지면 중단한다.

In [ ]:
EFFECT_STEPS = [
    ("zeroSampleEffect", "raw_baseline", "drop_zero_keep_weight"),
    ("informativeMassRenormalizationEffect", "drop_zero_keep_weight", "drop_zero_renormalized"),
    ("generatorMultiplicityEffect", "drop_zero_renormalized", "class_balanced_expanded"),
    ("rowCompressionEffect", "class_balanced_expanded", "cell_grouped"),
    ("totalGroupedChange", "raw_baseline", "cell_grouped"),
]


def fold_effect(left: str, right: str) -> dict[str, Any]:
    rows = []
    for fold in FOLD_RESULTS:
        left_mrr = fold["variants"][left]["physicalClassMetrics"]["mrr"]
        right_mrr = fold["variants"][right]["physicalClassMetrics"]["mrr"]
        delta = right_mrr - left_mrr
        direction = (
            "improved" if delta > TOLERANCES["metricEffect"]
            else "worse" if delta < -TOLERANCES["metricEffect"]
            else "equal"
        )
        rows.append({
            "validationGameId": fold["validationGameId"],
            "leftMrr": left_mrr,
            "rightMrr": right_mrr,
            "delta": delta,
            "direction": direction,
        })
    worst = min(rows, key=lambda row: (row["delta"], codepoint_key(row["validationGameId"])))
    return {
        "microClassMrrDelta": (
            EVALUATION_RESULTS[right]["physicalClass"]["micro"]["mrr"]
            - EVALUATION_RESULTS[left]["physicalClass"]["micro"]["mrr"]
        ),
        "macroClassMrrDelta": (
            EVALUATION_RESULTS[right]["physicalClass"]["macro"]["mrr"]
            - EVALUATION_RESULTS[left]["physicalClass"]["macro"]["mrr"]
        ),
        "improvedFoldCount": sum(row["direction"] == "improved" for row in rows),
        "equalFoldCount": sum(row["direction"] == "equal" for row in rows),
        "worseFoldCount": sum(row["direction"] == "worse" for row in rows),
        "worstFold": worst,
        "foldDeltas": rows,
    }


EFFECTS = {
    name: fold_effect(left, right)
    for name, left, right in EFFECT_STEPS
}
micro_step_sum = sum(
    EFFECTS[name]["microClassMrrDelta"]
    for name, _, _ in EFFECT_STEPS[:4]
)
macro_step_sum = sum(
    EFFECTS[name]["macroClassMrrDelta"]
    for name, _, _ in EFFECT_STEPS[:4]
)
assert abs(micro_step_sum - EFFECTS["totalGroupedChange"]["microClassMrrDelta"]) <= 1e-15
assert abs(macro_step_sum - EFFECTS["totalGroupedChange"]["macroClassMrrDelta"]) <= 1e-15

selected_multiplicity = dict(zip(
    class_frame.loc[class_frame["selectedClass"], "decisionKey"],
    class_frame.loc[class_frame["selectedClass"], "multiplicity"],
))
SEGMENTATION = {}
for variant in VARIANT_NAMES:
    rows = pd.DataFrame(HELD_OUT[variant]["physicalClass"])
    rows["selectedClassMultiplicity"] = rows["decisionKey"].map(selected_multiplicity)
    SEGMENTATION[variant] = []
    for multiplicity, segment in rows.groupby("selectedClassMultiplicity", sort=True):
        metrics = summarize_predictions(segment.to_dict(orient="records"))
        SEGMENTATION[variant].append({
            "selectedClassMultiplicity": int(multiplicity), **metrics
        })

PAIRLESS_DIAGNOSTIC = {
    variant: sorted(FULL_PAIRLESS[variant], key=codepoint_key)
    for variant in VARIANT_NAMES
}
assert len(PAIRLESS_DIAGNOSTIC["cell_grouped"]) == 2
display(pd.DataFrame([{
    "effect": name,
    **{key: value for key, value in result.items() if key != "foldDeltas"},
} for name, result in EFFECTS.items()]))
display(pd.DataFrame(SEGMENTATION["cell_grouped"]))

## 16. Baseline reproduction과 sanity gate

**왜 실행하는가:** attribution 전에 양 끝점이 기존 실험과 같은지 확인한다.

**입력:** V0 raw baseline과 V4 cell-grouped held-out 결과.

**계산:** integer count는 exact, 공개된 소수 metric은 6자리 반올림으로 비교한다. 수치-equivalence는 coefficient/score별 별도 tolerance를 쓴다.

**읽는 법:** binary logistic loss를 L(w; x, y)라 하면 x=0에서 L=log(2), ∇L=0, ∇²L=0이다. 따라서 zero row는 목적함수에 상수만 더하고 수학적 minimizer를 바꾸지 않는다. 유한 정밀도 solver의 종료점은 미세하게 다를 수 있으므로 V0↔V1은 coefficient/score tolerance와 exact rank로 비교한다. V3↔V4는 같은 class-level weight를 표현하므로 같은 방식으로 검증한다.

**보류 조건:** endpoint, leakage, convergence, equivalence gate 중 하나라도 실패하면 status는 blocked다.

In [ ]:
EXPECTED_V0_RAW = {
    "top1Count": 561,
    "top3Count": 886,
    "mrrRounded6": 0.645290,
    "meanRankRounded6": 2.961140,
}
EXPECTED_V4_CLASS = {
    "top1Count": 657,
    "top3Count": 1013,
    "mrrRounded6": 0.726652,
    "macroMrrRounded6": 0.702139,
    "meanRankRounded6": 2.085492,
}
v0_raw = EVALUATION_RESULTS["raw_baseline"]["rawCandidate"]
v4_class = EVALUATION_RESULTS["cell_grouped"]["physicalClass"]
V0_REPRODUCED = (
    v0_raw["micro"]["top1Count"] == EXPECTED_V0_RAW["top1Count"]
    and v0_raw["micro"]["top3Count"] == EXPECTED_V0_RAW["top3Count"]
    and round(v0_raw["micro"]["mrr"], 6) == EXPECTED_V0_RAW["mrrRounded6"]
    and round(v0_raw["micro"]["meanSelectedRank"], 6) == EXPECTED_V0_RAW["meanRankRounded6"]
)
V4_REPRODUCED = (
    v4_class["micro"]["top1Count"] == EXPECTED_V4_CLASS["top1Count"]
    and v4_class["micro"]["top3Count"] == EXPECTED_V4_CLASS["top3Count"]
    and round(v4_class["micro"]["mrr"], 6) == EXPECTED_V4_CLASS["mrrRounded6"]
    and round(v4_class["macro"]["mrr"], 6) == EXPECTED_V4_CLASS["macroMrrRounded6"]
    and round(v4_class["micro"]["meanSelectedRank"], 6) == EXPECTED_V4_CLASS["meanRankRounded6"]
)
v0_v1 = COMPARISONS["drop_zero_keep_weight-minus-raw_baseline"]
v3_v4 = COMPARISONS["cell_grouped-minus-class_balanced_expanded"]
ZERO_SAMPLE_EQUIVALENT = (
    v0_v1["maximumAbsoluteCoefficientDifference"] <= TOLERANCES["equivalentCoefficient"]
    and v0_v1["maximumAbsoluteValidationCandidateScoreDifference"] <= TOLERANCES["equivalentScore"]
    and v0_v1["physicalClassRankChangeCount"] == 0
)
ROW_COMPRESSION_EQUIVALENT = (
    v3_v4["maximumAbsoluteCoefficientDifference"] <= TOLERANCES["equivalentCoefficient"]
    and v3_v4["maximumAbsoluteValidationCandidateScoreDifference"] <= TOLERANCES["equivalentScore"]
    and v3_v4["physicalClassRankChangeCount"] == 0
)
ALL_FOLDS_CONVERGED = all(
    not fold["variants"][variant]["fit"]["convergenceWarnings"]
    for fold in FOLD_RESULTS for variant in VARIANT_NAMES
)
BASELINE_GATE = V0_REPRODUCED and V4_REPRODUCED
if not BASELINE_GATE:
    EXPERIMENT_STATUS = "blocked_by_baseline_mismatch"
elif ZERO_SAMPLE_EQUIVALENT and ROW_COMPRESSION_EQUIVALENT:
    EXPERIMENT_STATUS = "pair_weight_tradeoff_explained"
else:
    explained_micro = sum(abs(EFFECTS[name]["microClassMrrDelta"]) for name in [
        "informativeMassRenormalizationEffect", "generatorMultiplicityEffect"
    ])
    total_micro = abs(EFFECTS["totalGroupedChange"]["microClassMrrDelta"])
    EXPERIMENT_STATUS = (
        "pair_weight_tradeoff_partially_explained"
        if explained_micro > 0 and total_micro > 0
        else "pair_weight_tradeoff_not_explained"
    )

assert BASELINE_GATE
assert ZERO_SAMPLE_EQUIVALENT
assert ROW_COMPRESSION_EQUIVALENT
assert EXPERIMENT_STATUS == "pair_weight_tradeoff_explained"

## 17. Attribution 결과

**왜 실행하는가:** 전체 V4−V0 하락 중 어느 ordered step이 얼마를 차지했는지 정량화한다.

**입력:** 인접 variant의 physical-class micro/macro MRR delta.

**계산:** 같은 방향의 non-zero step을 total delta로 나눠 비율을 구한다.

**읽는 법:** 이는 이 순서에서의 설명 비율이며 완전한 인과·순서 독립 attribution이 아니다.

**보류 조건:** zero/compression step이 tolerance 밖이거나 두 weight step의 합이 total과 맞지 않으면 결론을 보류한다.

In [ ]:
total_micro_delta = EFFECTS["totalGroupedChange"]["microClassMrrDelta"]
total_macro_delta = EFFECTS["totalGroupedChange"]["macroClassMrrDelta"]
ATTRIBUTION = {
    "orderedDecomposition": True,
    "shapleyAttribution": False,
    "micro": {
        "zeroSampleEffect": EFFECTS["zeroSampleEffect"]["microClassMrrDelta"],
        "informativeMassRenormalizationEffect": EFFECTS[
            "informativeMassRenormalizationEffect"
        ]["microClassMrrDelta"],
        "generatorMultiplicityEffect": EFFECTS[
            "generatorMultiplicityEffect"
        ]["microClassMrrDelta"],
        "rowCompressionEffect": EFFECTS["rowCompressionEffect"]["microClassMrrDelta"],
        "totalGroupedChange": total_micro_delta,
    },
    "macro": {
        "zeroSampleEffect": EFFECTS["zeroSampleEffect"]["macroClassMrrDelta"],
        "informativeMassRenormalizationEffect": EFFECTS[
            "informativeMassRenormalizationEffect"
        ]["macroClassMrrDelta"],
        "generatorMultiplicityEffect": EFFECTS[
            "generatorMultiplicityEffect"
        ]["macroClassMrrDelta"],
        "rowCompressionEffect": EFFECTS["rowCompressionEffect"]["macroClassMrrDelta"],
        "totalGroupedChange": total_macro_delta,
    },
}
ATTRIBUTION["micro"]["informativeMassShareOfTotal"] = (
    ATTRIBUTION["micro"]["informativeMassRenormalizationEffect"] / total_micro_delta
)
ATTRIBUTION["micro"]["generatorMultiplicityShareOfTotal"] = (
    ATTRIBUTION["micro"]["generatorMultiplicityEffect"] / total_micro_delta
)
ATTRIBUTION["macro"]["informativeMassShareOfTotal"] = (
    ATTRIBUTION["macro"]["informativeMassRenormalizationEffect"] / total_macro_delta
)
ATTRIBUTION["macro"]["generatorMultiplicityShareOfTotal"] = (
    ATTRIBUTION["macro"]["generatorMultiplicityEffect"] / total_macro_delta
)
assert abs(ATTRIBUTION["micro"]["zeroSampleEffect"]) <= TOLERANCES["metricEffect"]
assert abs(ATTRIBUTION["micro"]["rowCompressionEffect"]) <= TOLERANCES["metricEffect"]
display(pd.DataFrame([
    {"scale": scale, **values} for scale, values in ATTRIBUTION.items()
    if scale in {"micro", "macro"}
]))

## 18. Source checksum 재검증과 결정적 report

**왜 실행하는가:** 실험이 source snapshot을 수정하지 않았고 결과 payload가 실행 순서와 무관함을 확인한다.

**입력:** 실행 전 digest와 모든 검증된 결과.

**계산:** 실행 후 digest를 비교하고 key-sorted JSON report를 무작위 sibling staging file로 atomic write한다.

**읽는 법:** run1/run2 report는 외부 검증에서 byte-identical이어야 한다.

**보류 조건:** timestamp/PID/사용자 절대 경로가 들어가거나 source digest가 변하면 결과를 보존하지 않는다.

In [ ]:
SOURCE_SHA256_AFTER = {name: sha256_file(path) for name, path in SOURCE_FILES.items()}
assert SOURCE_SHA256_AFTER == SOURCE_SHA256_BEFORE, "SOURCE_SNAPSHOT_MUTATED"

fold_summary = {
    "foldCount": 26,
    "allVariantsConvergedFoldCount": {
        variant: sum(
            not fold["variants"][variant]["fit"]["convergenceWarnings"]
            for fold in FOLD_RESULTS
        )
        for variant in VARIANT_NAMES
    },
    "iterationRange": {
        variant: {
            "minimum": min(fold["variants"][variant]["fit"]["iterations"] for fold in FOLD_RESULTS),
            "maximum": max(fold["variants"][variant]["fit"]["iterations"] for fold in FOLD_RESULTS),
        }
        for variant in VARIANT_NAMES
    },
    "gameIdLeakageCount": 0,
    "scalerFitScope": "raw_train_candidates_only",
    "sharedScalerAcrossVariants": True,
}
report = {
    "experimentReportSchemaVersion": 1,
    "experimentId": EXPERIMENT_ID,
    "status": EXPERIMENT_STATUS,
    "programGate": PROGRAM_GATE,
    "recommendation": (
        "The observed tradeoff is attributable to weight redistribution. Keep representation "
        "choice open and test the intended weighting policy next; do not freeze or deploy a model."
    ),
    "sourceSnapshot": {
        "relativeDirectory": DATASET_RELATIVE_DIR.as_posix(),
        "sha256": SOURCE_SHA256_BEFORE,
        "sourceUnchanged": True,
    },
    "environment": ACTUAL_ENVIRONMENT,
    "experimentCard": EXPERIMENT_CARD,
    "featureContract": {
        "featureSchemaVersion": FEATURE_SCHEMA_VERSION,
        "featureNames": FEATURE_NAMES,
        "forbiddenLeakageInputs": sorted(FORBIDDEN_FEATURES),
        "isWinnerUsage": "diagnostic_segmentation_only",
    },
    "modelConfig": MODEL_CONFIG,
    "tolerances": TOLERANCES,
    "dataSummary": {
        "gameCount": 26,
        "winnerGameCount": int(
            candidates.groupby("gameId", sort=False)["isWinner"].first().sum()
        ),
        "decisionCount": 1_158,
        "candidateCount": 39_375,
        "physicalClassCount": 26_403,
        "rawPairCount": 38_217,
        "groupedPairCount": 25_245,
        "zeroDifferencePairCount": 1_010,
        "selectedFeatureDuplicateDecisionCount": 670,
        "physicalPairlessDecisionCount": 2,
    },
    "workedExample": {
        **WORKED_SUMMARY,
        "variantWeightSummary": worked_weight_summary.to_dict(orient="records"),
    },
    "variantDefinitions": VARIANT_DEFINITIONS,
    "pairDiagnostics": PAIR_DIAGNOSTICS,
    "selectedClassMultiplicityDistribution": selected_multiplicity_distribution.to_dict(
        orient="records"
    ),
    "nonSelectedClassMultiplicityDistribution": nonselected_multiplicity_distribution.to_dict(
        orient="records"
    ),
    "evaluationResults": EVALUATION_RESULTS,
    "coefficientAndRankComparisons": COMPARISONS,
    "orderedEffects": EFFECTS,
    "attribution": ATTRIBUTION,
    "selectedMultiplicitySegmentation": SEGMENTATION,
    "pairlessDecisions": PAIRLESS_DIAGNOSTIC,
    "baselineReproduction": {
        "v0RawBaselineReproduced": V0_REPRODUCED,
        "v4CellGroupedReproduced": V4_REPRODUCED,
        "zeroSampleEquivalent": ZERO_SAMPLE_EQUIVALENT,
        "rowCompressionEquivalent": ROW_COMPRESSION_EQUIVALENT,
    },
    "foldSummary": fold_summary,
    "foldResults": FOLD_RESULTS,
    "contracts": {
        "allVariantsShareFoldAndScaler": True,
        "validationExcludedFromScalerPairsAndWeights": True,
        "cUsesFullTrainDecisionCount": True,
        "gameIdLeakageCount": 0,
        "physicalClassPrimaryDiagnostic": True,
        "allFoldsConverged": ALL_FOLDS_CONVERGED,
        "sourceSnapshotUnchanged": True,
        "productionModelUpdated": False,
        "runtimeIntegration": False,
        "gameplayStrengthClaimed": False,
    },
    "limitations": [
        "Only 26 games are observed.",
        "Only 3 winner games are present; winner performance is reference-only.",
        "Ordered deltas are not order-independent causal or Shapley attribution.",
        "Offline ranking metrics do not establish gameplay strength or win rate.",
    ],
}
report_text = json.dumps(
    report, ensure_ascii=False, indent=2, sort_keys=True, allow_nan=False
) + "\n"
assert "timestamp" not in report_text.lower()
assert '"pid"' not in report_text.lower()
assert str(ROOT) not in report_text

In [ ]:
def same_inode(left: Path, right: Path) -> bool:
    left_stat, right_stat = left.stat(), right.stat()
    return (left_stat.st_dev, left_stat.st_ino) == (right_stat.st_dev, right_stat.st_ino)


def atomic_write_text(output_path: Path, text: str, protected_sources: list[Path]) -> None:
    """Write through an exclusive random sibling and atomically replace the report."""
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_real = output_path.resolve(strict=False)
    for source_path in protected_sources:
        assert output_real != source_path.resolve(strict=True), "REPORT_SOURCE_COLLISION"
        if os.path.lexists(output_path):
            assert not output_path.is_symlink(), "REPORT_OUTPUT_SYMLINK_NOT_ALLOWED"
            assert not same_inode(output_path, source_path), "REPORT_OUTPUT_INODE_COLLISION"
    descriptor, staging_name = tempfile.mkstemp(
        prefix=f".{output_path.name}.", suffix=".tmp", dir=output_path.parent
    )
    staging_path = Path(staging_name)
    descriptor_stat = os.fstat(descriptor)
    try:
        with os.fdopen(descriptor, "w", encoding="utf-8", newline="\n") as handle:
            handle.write(text)
            handle.flush()
            os.fsync(handle.fileno())
        staging_stat = staging_path.lstat()
        assert not staging_path.is_symlink()
        assert (staging_stat.st_dev, staging_stat.st_ino) == (
            descriptor_stat.st_dev, descriptor_stat.st_ino
        )
        assert all(not same_inode(staging_path, source) for source in protected_sources)
        os.replace(staging_path, output_path)
    finally:
        if staging_path.exists():
            staging_path.unlink()


REPORT_PATH = Path(os.environ.get(
    "PLACEMENT_PAIR_WEIGHT_REPORT",
    "/private/tmp/placement-pair-weight-report.json",
))
atomic_write_text(REPORT_PATH, report_text, list(SOURCE_FILES.values()))
REPORT_SHA256 = hashlib.sha256(report_text.encode("utf-8")).hexdigest()
assert sha256_file(REPORT_PATH) == REPORT_SHA256
display(pd.DataFrame([{
    "status": EXPERIMENT_STATUS,
    "programGate": PROGRAM_GATE,
    "reportSha256": REPORT_SHA256,
    "sourceSnapshotUnchanged": SOURCE_SHA256_AFTER == SOURCE_SHA256_BEFORE,
}]))

## 19. 관측, 해석, 결정과 다음 gate

**관측:** V0/V1과 V3/V4의 held-out rank는 같고, metric 하락은 두 weight 재분배 단계에서만 발생한다.

**해석:** zero row 자체나 물리적으로 같은 row의 저장 개수보다, selected alias가 소비하던 informative mass를 되돌리는 변화와 non-selected generator multiplicity를 class-balanced하게 만드는 변화가 이 순서의 delta를 설명한다.

**결정:** 상태는 `pair_weight_tradeoff_explained`, 프로그램 gate는 `continue_experimenting`이다. 이 결과만으로 representation을 freeze하거나 production 모델을 만들지 않는다.

**아직 말할 수 없는 것:** gameplay 성능, 승률, 더 큰 데이터 일반화, 3개뿐인 winner game의 독립 성능.

**다음 gate:** 동일 representation에서 의도한 weighting policy를 먼저 결정한 뒤, 별도 승인된 실험으로 데이터 규모와 winner segment를 보강한다.

In [ ]:
assert EXPERIMENT_STATUS in EXPERIMENT_CARD["possibleStatuses"]
assert EXPERIMENT_STATUS == "pair_weight_tradeoff_explained"
assert PROGRAM_GATE == "continue_experimenting"
assert report["contracts"]["productionModelUpdated"] is False
assert report["contracts"]["runtimeIntegration"] is False
{
    "status": EXPERIMENT_STATUS,
    "primaryMicroDelta": ATTRIBUTION["micro"]["totalGroupedChange"],
    "primaryMacroDelta": ATTRIBUTION["macro"]["totalGroupedChange"],
    "nextGate": PROGRAM_GATE,
}